In [1]:
import sys
sys.path.insert(0,'../g3algo/')
from g3groupfinder import giantmodel, decayexp, sigmarange
import foftools as fof
import iterativecombination as ic
from smoothedbootstrap import smoothedbootstrap as sbs
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator
from matplotlib import rcParams
from scipy.optimize import curve_fit
from center_binned_stats import center_binned_stats
from scipy.stats import ks_2samp as kstest
from scipy.interpolate import interp1d
rcParams['axes.labelsize'] = 9
rcParams['xtick.labelsize'] = 9
rcParams['ytick.labelsize'] = 9
rcParams['legend.fontsize'] = 9
rcParams['font.family'] = 'sans-serif'
rcParams['grid.color'] = 'k'
rcParams['grid.linewidth'] = 0.2
my_locator = MaxNLocator(6)
singlecolsize = (3.3522420091324205, 2.0717995001590714)
doublecolsize = (7.100005949910059, 4.3880449973709)
HUBBLE_CONST = 70.
import survey_volume as sv

from scipy.spatial.distance import euclidean
def hellinger2(p, q):
    return euclidean(np.sqrt(p), np.sqrt(q)) / np.sqrt(2)

In [2]:
# %matplotlib inline
# df1 = pd.read_csv("ECOdata_G3catalog_luminosity_assubmitted.csv").set_index('name')
# df2 = pd.read_csv("ECOdata_G3catalog_luminosity.csv").set_index('name').sort_index()
# df1 = df1.loc[df2.index,:].sort_index()

# sel = (df1.absrmag>-17.33) & (df2.absrmag_withoutparens<=-17.33)

# plt.figure()
# plt.plot(df1.absrmag[sel],df2.absrmag_withoutparens[sel],'k.')
# plt.plot(df1.absrmag[sel],df2[sel].rmag_withoutparens + 5 - 5*np.log10(df2[sel].cz/70.*1e6) - df2[sel].extr,'g*')
# tx = np.linspace(-17.33,-17.28,3)
# plt.plot(tx,tx)
# plt.show()

# plt.figure()
# plt.plot(df1.absrmag,df2.absrmag_withoutparens-df1.absrmag,'k.')
# tx = np.linspace(-24,-17.28,3)
# plt.plot(tx,tx*0)
# plt.show()

# print(df2[sel].cz.describe())

In [3]:
ecodata = pd.read_csv("ECOdata_G3catalog_luminosity.csv")
ecodata = ecodata[(ecodata.cz<7470)&(ecodata.cz>2530)]
print(len(ecodata[ecodata.absrmag<=-17.33]))
print(len(ecodata[ecodata.absrmag<-19.5]))
#print(len(ecodata[ecodata.absrmag>=-19.5]))
print(len(ecodata[(ecodata.absrmag<=-17.33)&(ecodata.g3grpcz_l>3000)&(ecodata.g3grpcz_l<7000)]))

12771
4714
9640


/tmp/ipykernel_786273/2740200415.py:1: DtypeWarning: Columns (70,83) have mixed types. Specify dtype option on import or set low_memory=False.
  ecodata = pd.read_csv("ECOdata_G3catalog_luminosity.csv")


In [4]:
sv.comoving_volume(130.05,237.45,-1,49.85,2530/3e+5,7470/3e5,100,0.3,0.7)

191936.07612395773

In [5]:
resdata = pd.read_csv("RESOLVEdata_G3catalog_luminosity.csv")
#resdata = resdata[(resdata.cz<7250)&(resdata.cz>4250)]
inlum = ((resdata.f_b>0)&(resdata.absrmag<=-17.)|((resdata.f_a>0)&(resdata.absrmag<=-17.33)))
print(len(resdata[inlum]))
print(len(resdata[resdata.absrmag<-19.5]))
print(len(resdata[(inlum)&(resdata.g3grpcz_l>=4500)&(resdata.g3grpcz_l<=7000)]))

1690
577
1457


In [ ]:
min(resdata.cz),max(resdata.cz)

In [ ]:
resdata = pd.read_csv("RESOLVEdata_G3catalog_luminosity.csv")
len(resdata[resdata.fl_insample>0])

In [ ]:
sv.comoving_volume(131.25,236.25,0,5,4250/3e+5,7250/3e5,100,0.3,0.7) # RESOLVE-A

In [ ]:
sv.comoving_volume(-30,45,-1.25,1.25,4250/3e+5,7250/3e5,100,0.3,0.7) # RESOLVE-b

In [ ]:
resdata = pd.read_csv("RESOLVEdata_080822.csv")
len(resdata[(resdata.fl_insample>0)])

In [ ]:
inlum = ((resdata.f_b>0) & (resdata.absrmag<= -17)) | ((resdata.f_a>0) & (resdata.absrmag <= -17.33)) 
len(resdata[inlum])

In [ ]:
insample = inlum & (resdata.grpcz>=4500) & (resdata.grpcz<=7000) 
len(resdata[insample])

what galaxy is missing?

In [ ]:
from scipy.io import readsav

In [ ]:
cielo = readsav('/srv/one/resolve/database_internal/merged_idl_catalog/stable/resolvecatalog.dat')
cielophot = readsav('/srv/one/resolve/database_internal/merged_idl_catalog/stable/resolvecatalogphot.dat')

In [ ]:
cielo_absrmag = np.float64(cielophot['absmagr'])
cielo_fa = np.float64(cielo['inspring'])
cielo_fb = np.float64(cielo['infall'])
cielo_grpcz = np.float64(cielo['groupcz'])
cielo_name = np.array([x.decode('utf-8') for x in cielo['name']])

In [ ]:
cielo_inlum = ((cielo_fb>0) & (cielo_absrmag<=-17)) | ((cielo_fa>0) & (cielo_absrmag<=-17.33))

In [ ]:
cielo_names_1671 = cielo_name[cielo_inlum]

In [ ]:
resdata = pd.read_csv("RESOLVEdata_080822.csv")
inlum = ((resdata.f_b>0) & (resdata.absrmag<=-17)) | ((resdata.f_a>0) & (resdata.absrmag <= -17.33)) 
resdata = resdata[inlum]

In [ ]:
wiki_names_1670 = resdata.name.to_numpy()

In [ ]:
wiki_names_1670

In [ ]:
for nm in cielo_names_1671:
    if nm not in wiki_names_1670:
        print(nm)

In [ ]:
print(len(cielo_names_1671),len(wiki_names_1670))

In [ ]:
resdata = pd.read_csv("RESOLVEdata_080822.csv")
resdata[resdata.name=='rf0725'].absrmag

In [ ]:
cielo_absrmag[cielo_name=='rf0725']

In [ ]:
resdata.absrmag